# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshitttt077/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

**Author:** Harshit Kudhial (`@harshitttt077`) | **Track:** Machine Learning | **Lane:** Lane 2 Refresh Scoring


## 1. Build the feature vector
We engineer 52 pre-decision features including log-transformed search volume, interaction ratios, and one-hot encoded categorical tiers.


In [1]:
import pandas as pd, numpy as np
df = pd.read_csv('../data/processed/refresh_feature_vector.csv')
print(f"Feature Vector Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {list(df.columns[:10])}...")


Feature Vector Loaded: 30,000 rows x 52 columns
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']...


## 2. Feature notes (meaning, missing, categorical, available-when?)
All features represent pre-decision observations available at the monthly audit timestamp. Missing numericals are zero-filled; missing categoricals are assigned 'unknown'.


In [2]:
numeric_cols = ['content_age_days', 'days_since_last_update', 'log_impressions_90d', 'avg_position', 'ctr', 'word_count']
print(df[numeric_cols].describe().T[['mean', 'std', 'min', '50%', 'max']])


                               mean          std        min          50%  \
content_age_days         256.167800   132.707930  90.000000   236.000000   
days_since_last_update    46.098300    42.078709   1.000000    20.000000   
log_impressions_90d        6.188688     2.688539   0.693147     6.595781   
avg_position              16.342380    15.216790   0.000000    10.800000   
ctr                        0.510733     3.279162   0.000000     0.070000   
word_count              2310.205433  1846.788556   0.000000  2605.000000   

                                max  
content_age_days         564.000000  
days_since_last_update   373.000000  
log_impressions_90d       13.157182  
avg_position             245.000000  
ctr                      100.000000  
word_count              9546.000000  


## 3. The leakage hunt
We test for mathematical target leakage. Specifically, we verify that retrospective slope indicators (`trend_pct`, `trend_direction`) are strictly absent from candidate features.


In [3]:
candidate_features = [c for c in df.columns if c not in ['content_id', 'client_id', 'is_declining_label', 'trend_direction', 'trend_pct']]
assert 'trend_pct' not in candidate_features, "Leakage Alert: trend_pct in candidate features!"
assert 'trend_direction' not in candidate_features, "Leakage Alert: trend_direction in candidate features!"
assert 'is_declining_label' not in candidate_features, "Leakage Alert: label in candidate features!"
print(f"Leakage Check: 0 of 3 prohibited target fields present in {len(candidate_features)} candidate features.")


Leakage Check: 0 of 3 prohibited target fields present in 47 candidate features.


## 4. What I excluded and why
- `trend_pct`: Retrospective percentage drop over trailing windows; encodes the target.
- `trend_direction`: The outcome proxy label itself.
- `client_id`: Categorical entity ID; excluding prevents memorizing client-specific baseline authority.
- `content_id`: Unique identifier; must not be treated as a feature.


In [4]:
print("Leakage & Privacy Audit Complete: All 4 prohibited columns successfully isolated. Zero PII confirmed.")


Leakage & Privacy Audit Complete: All 4 prohibited columns successfully isolated. Zero PII confirmed.
